# SynthID-Text

Short notebook to test synthid-text with llama3.1-8B

## Generate Text with SynthID-Text Watermark

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import LogitsProcessorList
from transformers import SynthIDTextWatermarkLogitsProcessor
from os import getenv
from transformers import BitsAndBytesConfig

# 4-Bit-Quantisierung konfigurieren
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

# 1. Modell und Tokenizer laden
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN = getenv("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN,
    clean_up_tokenization_spaces=False
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    torch_dtype="auto",
    device_map="mps",
    quantization_config=quantization_config
)

# 2. SynthID LogitsProcessor konfigurieren
# Ein geheimer Schlüssel (keys) steuert die mathematische Pseudozufallsfunktion
synthid_processor = SynthIDTextWatermarkLogitsProcessor(
    keys=[1234, 5678, 9012],  # Pseudozufalls-Schlüssel
    sampling_table_size=1024,  # Größe der Hash-Tabelle für Logit-Shift
    ngram_len=5,
    sampling_table_seed=42,  # Seed für Determinismus
    context_history_size=5,  # Entspricht der Kontextlänge (ngram_len)
    device="mps"
)

# In die LogitsProcessorList von Hugging Face einreihen
logits_processor = LogitsProcessorList([synthid_processor])

# 3. Input vorbereiten
prompt = "Erkläre kurz, was ein Quantencomputer ist."
messages = [{"role": "user", "content": prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

# 4. Text generieren (mit eingebettetem Wasserzeichen)
output_ids = model.generate(
    **inputs,  # <--- Hier mit ** entpacken
    max_new_tokens=400,
    do_sample=True,
    temperature=0.7,
    logits_processor=logits_processor,
)

# 5. Output dekodieren
input_length = inputs["input_ids"].shape[1]
generated_text = tokenizer.decode(
    output_ids[0][input_length:], skip_special_tokens=True
)

print("Generierter Text (mit SynthID Wasserzeichen):\n")
print(generated_text)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Generierter Text (mit SynthID Wasserzeichen):

Ein Quantencomputer ist ein Computer, der auf den Prinzipien der Quantenmechanik basiert und die Fähigkeit besitzt, bestimmte Berechnungen viel schneller als traditionelle Computer￣￣￣ auszuführen. Er nutzt die Eigenschaften von Teilchen im Quantenzustand, wie z.B. Elektronen, um komplexe Berechnungen parallel und gleichzeitig auszuführen.

Ein traditioneller Computer verwendet Bit (Binary digit), um Informationen zu verarbeiten. Ein Bit kann nur entweder 0 oder 1 sein. Ein Quantencomputer hingegen verwendet Quantenbits (Qubits), die eine Kombination von 0 und 1 sein können, bis ein Messung vorgenommen wird. Dies ermöglicht es, viele Berechnungen gleichzeitig auszuführen, was die Geschwindigkeit eines Quantencomputers stark erhöht.

Einige der wichtigsten Eigenschaften eines Quantencomputers sind:

- Parallele Berechnungen: Ein Quantencomputer kann viele Berechnungen gleichzeitig ausführen, was die Lösung von komplexen Problemen erleichtert

## Check generated Text

In [7]:
import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import (
    AutoTokenizer,
    BayesianDetectorModel,
    PretrainedConfig,
    SynthIDTextWatermarkDetector,
    SynthIDTextWatermarkLogitsProcessor,
)

# 1. Class-Patch für transformers Kompatibilität
if not hasattr(BayesianDetectorModel, "all_tied_weights_keys"):
    BayesianDetectorModel.all_tied_weights_keys = []

model_hub_id = "joaogante/dummy_synthid_detector"

# 2. Config manuell laden und reparieren
config = PretrainedConfig.from_pretrained(model_hub_id)

if isinstance(config.watermarking_config, list):
    # Die Liste in das darin enthaltene Dictionary auflösen
    config.watermarking_config = config.watermarking_config[0]

# 3. Detektor-Modell mit der reparierten Config laden
detector_model = BayesianDetectorModel(config)

weights_path = hf_hub_download(repo_id=model_hub_id, filename="model.safetensors")
state_dict = load_file(weights_path)
detector_model.load_state_dict(state_dict)
detector_model.to("mps")

# 4. LogitsProcessor aus der korrigierten Config erstellen
synthid_processor = SynthIDTextWatermarkLogitsProcessor(
    **config.watermarking_config,
    device="mps",
)

# 5. SynthID-Detektor instanziieren
detector = SynthIDTextWatermarkDetector(
    detector_module=detector_model,
    logits_processor=synthid_processor,
    tokenizer=tokenizer,
)

# 6. Text prüfen
test_input = tokenizer(generated_text, return_tensors="pt").input_ids.to("mps")
outputs = detector(test_input)

print("\n--- SynthID Prüfergebnis ---")
print(outputs)


--- SynthID Prüfergebnis ---
(tensor([2.3040e-18], device='mps:0', grad_fn=<SigmoidBackward0>),)


In [8]:
# Output-Tensor entpacken
probability = outputs[0].item()

print(f"Wasserzeichen-Wahrscheinlichkeit: {probability:.2%}")
# Ausgabe: Wasserzeichen-Wahrscheinlichkeit: 0.00%

if probability > 0.8:
    print("Ergebnis: Wasserzeichen vorhanden")
else:
    print("Ergebnis: Kein Wasserzeichen erkannt")

Wasserzeichen-Wahrscheinlichkeit: 0.00%
Ergebnis: Kein Wasserzeichen erkannt
